Lecture simple d'un fichier texte en Python

In [5]:
import os
with open('../data/text_samples/intro_rag.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    print(text)

L’ingestion est la première étape d’un projet RAG.
Elle permet au modèle de comprendre des données externes.
Un bon parsing améliore la qualité des réponses.



In [7]:
from langchain_core.documents import Document
doc = Document(page_content=text, metadata={"source": "../data/text_samples/intro_rag.txt", 
                                            "format": "txt"})

In [8]:
print(doc)

page_content='L’ingestion est la première étape d’un projet RAG.
Elle permet au modèle de comprendre des données externes.
Un bon parsing améliore la qualité des réponses.
' metadata={'source': '../data/text_samples/intro_rag.txt', 'format': 'txt'}


In [12]:
print("contenu", doc.page_content[:80])
print("metadata", doc.metadata)

contenu L’ingestion est la première étape d’un projet RAG.
Elle permet au modèle de comp
metadata {'source': '../data/text_samples/intro_rag.txt', 'format': 'txt'}


Utilisation de textLoaders pour lire un fichier texte et créer un document LangChain

In [16]:
from langchain_community.document_loaders import TextLoader

/tmp/ipykernel_28174/2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Créaction du chargeur de texte 

In [17]:
loader = TextLoader('../data/text_samples/intro_rag.txt', encoding='utf-8')

In [18]:
docs = loader.load()

In [19]:
type(docs), len(docs)

(list, 1)

In [21]:
docs[0].metadata

{'source': '../data/text_samples/intro_rag.txt'}

Création du second fichier texte

In [23]:
sample_text = """
Les fichiers texte sont la forme la plus simple de données à ingérer.
Mais dans un projet réel, tu auras souvent des PDF, des CSV, ou des documents Word.
L'idée est de tout standardiser sous forme de Documents pour faciliter le traitement en aval.
"""

with open('../data/text_samples/ingestion_2.txt', 'w', encoding='utf-8') as f:
    f.write(sample_text)
    print("nouveau fichier texte créé : ../data/text_samples/ingestion_2.txt")

nouveau fichier texte créé : ../data/text_samples/ingestion_2.txt


Démonstation pour plusieurs fichiers texte

In [26]:
from langchain_community.document_loaders import DirectoryLoader
multi_loader = DirectoryLoader('../data/text_samples', glob="*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})  
documents = multi_loader.load()

print(f"Nombre de documents chargés : {len(documents)}")
for d in documents:
    print(d.metadata)

Nombre de documents chargés : 3
{'source': '../data/text_samples/ingestion_2.txt'}
{'source': '../data/text_samples/texte_exemple_rag.txt'}
{'source': '../data/text_samples/intro_rag.txt'}


Ajout de metadata personnalisée pour chaque document

In [27]:
for doc in documents:
    if "intro" in doc.metadata["source"]:
        doc.metadata["theme"] = "introduction"
    elif "ingestion" in doc.metadata["source"]:
        doc.metadata["theme"] = "directoryLoader"

    doc.metadata["author"] = "Lady"
    doc.metadata["format"] = "text"
    doc.metadata["version"] = "v1.0"

for doc in documents:
    print(doc.metadata)

{'source': '../data/text_samples/ingestion_2.txt', 'theme': 'directoryLoader', 'author': 'Lady', 'format': 'text', 'version': 'v1.0'}
{'source': '../data/text_samples/texte_exemple_rag.txt', 'author': 'Lady', 'format': 'text', 'version': 'v1.0'}
{'source': '../data/text_samples/intro_rag.txt', 'theme': 'introduction', 'author': 'Lady', 'format': 'text', 'version': 'v1.0'}


In [30]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('../data/text_samples/texte_exemple_rag.txt', encoding='utf-8')
documents = loader.load()

print(documents[0].page_content[:300])


GUIDE COMPLET – RAG & PYTHON

Le Retrieval-Augmented Generation (RAG) est une architecture moderne qui permet d'améliorer 
de manière spectaculaire la qualité et la fiabilité des réponses générées par les modèles 
de langage. L'idée cent


In [34]:
def clean_text(text :str) -> str:
    import re
    text = re.sub(r'\s+', ' ', text)  # Remplace les espaces multiples par un seul espace
    text = text.strip()  # Supprime les espaces au début et à la fin du texte
    return text

In [32]:
def clean_documents(documents):
    cleaned_documents = []
    for doc in documents:
        if not doc.page_content:
            doc.page_content = ""
        else:
            doc.page_content = clean_text(doc.page_content)
        cleaned_documents.append(doc)
    return cleaned_documents

In [35]:
cleaned_docs = clean_documents(documents)

print("Avant nettoyage :", documents[0].page_content[:300])
print("Après nettoyage :", cleaned_docs[0].page_content[:300])

Avant nettoyage : ============================== GUIDE COMPLET – RAG & PYTHON ============================== Le Retrieval-Augmented Generation (RAG) est une architecture moderne qui permet d'améliorer de manière spectaculaire la qualité et la fiabilité des réponses générées par les modèles de langage. L'idée centrale
Après nettoyage : ============================== GUIDE COMPLET – RAG & PYTHON ============================== Le Retrieval-Augmented Generation (RAG) est une architecture moderne qui permet d'améliorer de manière spectaculaire la qualité et la fiabilité des réponses générées par les modèles de langage. L'idée centrale


Découper les documents (Document Splitters)

Ce qui fait un bon chunk\
    Un bon chunk :
    - Il doit contenir une seule idée principale,
    - Avec un minimum de contexte pour éviter les erreurs d’interprétation,
    - Pas trop long (risque de dépasser le contexte du modèle),
    - Pas trop court (le modèle manque d'information)

Le splitter utilisé dans 95% des projets : **RecursiveCharacterTextSplitter**

1. doubles sauts de ligne

2. sauts de ligne

3. points

4. espaces

5. seulement en dernier recours : caractère par caractère

In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # taille idéale d’un chunk en caractères
    chunk_overlap=150,    # chevauchement entre les chunks
    separators=["\n\n", "\n", ".", " ", ""]  # ordre logique de découpe
)

In [38]:
chunks = splitter.split_documents(cleaned_docs)

In [40]:
# Inspecter les chunks,
print(f"Nombre total de chunks : {len(chunks)}")
print(f" chunk1: {chunks[0].page_content}")
print(f" chunk2: {chunks[1].page_content}")
print(f" chunk3: {chunks[2].page_content}")

Nombre total de chunks : 23
 chunk1: ============================== GUIDE COMPLET – RAG & PYTHON ============================== Le Retrieval-Augmented Generation (RAG) est une architecture moderne qui permet d'améliorer de manière spectaculaire la qualité et la fiabilité des réponses générées par les modèles de langage. L'idée centrale est simple : au lieu de laisser le modèle "inventer" du contenu ou s'appuyer uniquement sur sa mémoire interne, on lui fournit des documents pertinents récupérés dynamiquement avant la génération finale. Dans un pipeline RAG classique, plusieurs étapes sont essentielles : 1. INGESTION DES DONNÉES L’ingestion consiste à récupérer les données depuis plusieurs sources (PDF, TXT, DOCX, CSV, pages web, bases SQL, APIs, etc.). Chaque donnée brute doit être lue, transformée, nettoyée et convertie dans un format cohérent. En Python, l’ingestion se fait souvent avec LangChain, LlamaIndex ou des scripts personnalisés. 2
 chunk2: . En Python, l’ingestion se fait so